## 1. Introduction & Conceptual Map

To understand how these four tests relate, it helps to view them through two distinct lenses: **Assumptions about the data** (Parametric vs. Non-parametric) and **Study Design** (Independent vs. Paired). 

Furthermore, we can view simple group comparisons as a special case of a broader generalized linear model, which is where the Wald test comes in.

* **Student’s t-test:** The baseline. A parametric test used to compare the means of two *independent* groups, assuming the data is normally distributed.
* **Mann-Whitney U test:** The non-parametric alternative to the independent Student's t-test. Used when the data violates the normality assumption. It compares mean *ranks* rather than raw means.
* **Wilcoxon signed-rank test:** The non-parametric alternative to the *paired* t-test. Used when comparing two *dependent* samples (e.g., before and after measurements) that are not normally distributed.
* **Wald test:** The generalization. A broad parametric test used in statistical modeling to determine if a parameter (like a regression coefficient) is significantly different from zero. An independent Student's t-test is mathematically equivalent to a Wald test performed on a simple linear regression!

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf

np.random.seed(42)

## 2. Independent Samples: Student's t-test vs. Mann-Whitney U test

Let's generate some data to represent two distinct, independent groups (e.g., Treatment A and Treatment B).

In [ ]:
# Generate normally distributed data for Student's t-test
group_A_normal = np.random.normal(loc=50, scale=10, size=30)
group_B_normal = np.random.normal(loc=60, scale=10, size=30)

# Generate skewed (non-normal) exponential data for Mann-Whitney test
group_A_skewed = np.random.exponential(scale=1/0.1, size=30)
group_B_skewed = np.random.exponential(scale=1/0.05, size=30)

# 2.1 The Parametric Approach: Student's t-test
# equal_var=True forces the classic Student's test over Welch's
t_stat, p_val_t = stats.ttest_ind(group_A_normal, group_B_normal, equal_var=True)
print(f"Student's t-test:\n  t-statistic = {t_stat:.4f}\n  p-value = {p_val_t:.4e}\n")

# 2.2 The Non-Parametric Approach: Mann-Whitney U test
mw_stat, p_val_mw = stats.mannwhitneyu(group_A_skewed, group_B_skewed, alternative='two-sided')
print(f"Mann-Whitney U test:\n  U-statistic = {mw_stat:.4f}\n  p-value = {p_val_mw:.4e}")

**Relationship:** The Mann-Whitney U test answers the same research question as the Student's t-test (are these two distinct groups different?) but sacrifices a small amount of statistical power to gain robustness against non-normal distributions and outliers.

---

## 3. Dependent Samples: Wilcoxon signed-rank test

Now, assume the data isn't from two separate groups, but rather the *same* group measured twice (e.g., patients' blood pressure before and after a drug).

In [ ]:
# Generate paired data (Before and After)
before_treatment = np.random.normal(loc=120, scale=15, size=30)
# The drug lowers BP, but with some random noise
after_treatment = before_treatment - np.random.normal(loc=10, scale=5, size=30)

# Wilcoxon signed-rank test
w_stat, p_val_w = stats.wilcoxon(before_treatment, after_treatment)
print(f"Wilcoxon signed-rank test:\n  W-statistic = {w_stat:.4f}\n  p-value = {p_val_w:.4e}")

**Relationship:** Just as Mann-Whitney relates to the *independent* t-test, the Wilcoxon signed-rank test is the non-parametric sibling to the *paired* t-test.

---

## 4. The Generalization: Student's t-test vs. Wald test

This is where the magic happens. A standard Student's t-test is actually just a very specific application of a linear model. If we run a linear regression predicting our outcome based on group membership, the Wald test on the group coefficient yields the exact same conclusions as a t-test.

In [ ]:
# 1. Combine our normal data into a pandas DataFrame
df = pd.DataFrame({
    'value': np.concatenate([group_A_normal, group_B_normal]),
    'group': ['A'] * 30 + ['B'] * 30
})

# 2. Run a simple linear regression: value ~ group
model = smf.ols('value ~ C(group)', data=df).fit()

# View the coefficients table. Look closely at the t-value for "C(group)[T.B]"
print(model.summary().tables[1])
print("\n" + "-"*50 + "\n")

# 3. Perform a Wald test on the "group B" coefficient 
wald_res = model.wald_test('C(group)[T.B] = 0')

# 4. Compare the p-values!
print(f"P-value from Student's t-test: {p_val_t:.8f}")
print(f"P-value from Wald test:        {wald_res.pvalue:.8f}")

**Relationship:** The t-test statistic squared ($t^2$) is equal to the Wald test statistic (which follows a Chi-square distribution with 1 degree of freedom in this scenario). The Wald test is the "parent" concept; it evaluates parameters in complex models (like logistic regression or Poisson regression) where a simple t-test is no longer applicable.

## Summary Table

| Test | Parametric? | Group Design | Primary Use Case |
| :--- | :--- | :--- | :--- |
| **Student's t-test** | Yes | Independent | Comparing means of two distinct, normally distributed groups. |
| **Mann-Whitney U** | No | Independent | Comparing distributions (ranks) of two distinct, skewed groups. |
| **Wilcoxon signed-rank**| No | Paired | Comparing median differences of repeated measures on the same group. |
| **Wald test** | Yes | Any (Model-based) | Testing if a parameter/coefficient in a statistical model differs from zero. |